# TP 2 — Construire l'auto-attention pas à pas
Objectif : partir d'une simple moyenne des tokens précédents et arriver à une tête qui apprend elle-même quels tokens doivent contribuer.

*Invariant à suivre* : une ligne de poids contient la contribution des tokens passés au calcul du token courant, et sa somme vaut toujours 1.

**Livrable** : un notebook contenant les quatre versions, les tests `torch.allclose`, la matrice `wei[0]` et une explication des formes B, T, C, d_h.

In [20]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## Étape 1 : agréger par produit matriciel

In [21]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)

b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


> **Questions d'interprétation** — Sans Python, calculez c₂. Pourquoi la somme de chaque ligne de `a` vaut-elle 1 ?

**Réponses**
- Avec `b = [[2, 7], [6, 4], [6, 5]]` : **c₂** (2ᵉ ligne) = ½·b₁ + ½·b₂ = ((2+6)/2, (7+4)/2) = **(4 ; 5,5)**. Pour la 3ᵉ ligne : ((2+6+6)/3, (7+4+5)/3) = (4,67 ; 5,33). Cela correspond à la sortie.
- **Somme des lignes = 1** : chaque ligne de `tril(ones)` est divisée par sa propre somme (`a / a.sum(1, keepdim=True)`). La ligne t contient donc t poids égaux à 1/t. C'est une **moyenne** des vecteurs bᵢ visibles, et non une somme qui grossirait avec t.

## Étape 2 : moyenner uniquement le passé

In [22]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]  # forme ?
        xbow[b, t] = torch.mean(xprev, dim = 0)
print(xprev)

tensor([[ 1.6455, -0.8030],
        [ 1.3514, -0.2759],
        [-1.5108,  2.1048],
        [ 2.7630, -1.7465],
        [ 1.4516, -1.5103],
        [ 0.8212, -0.2115],
        [ 0.7789,  1.5333],
        [ 1.6097, -0.4032]])


> **Questions d'interprétation** — Donnez les formes de `xprev` pour t=0, t=3 et t=7. Pourquoi la moyenne porte-t-elle sur `dim=0` ?

**Réponses**
- `xprev = x[b, :t+1]` a la forme **(t+1, C)** : t=0 → **(1, 2)**, t=3 → **(4, 2)**, t=7 → **(8, 2)**.
- **`dim=0`** : une fois `b` fixé, l'axe 0 de `xprev` est l'axe des **tokens** (le temps) et l'axe 1 celui des composantes. On veut moyenner *les tokens* pour obtenir un vecteur de taille C, qui va dans `xbow[b, t]`.



## Étapes 3–4 : vectoriser
La matrice triangulaire reproduit la moyenne causale ; le masque −∞ suivi de Softmax produit exactement les mêmes poids.

### Version 2 : produit matriciel

In [23]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
print(torch.allclose(
    xbow, xbow2, atol=1e-6
))

True


### Version 3 : masque + Softmax

In [24]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print(torch.allclose(
    xbow, xbow3, atol=1e-6
))
print(wei)

True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


> **Questions d'interprétation** — Pourquoi `wei @ x` fonctionne-t-il avec une matrice (T, T) et un tenseur (B, T, C) ? Que devient Softmax aux emplacements contenant −∞ ?

**Réponses**
- **Broadcasting du produit matriciel** : `@` multiplie les deux **dernières** dimensions et traite les précédentes comme des dimensions de batch. `wei` de forme `(T, T)` est répété implicitement sur les B séquences : `(T, T) @ (B, T, C)` → `(B, T, T) @ (B, T, C)` → **`(B, T, C)`**. Les mêmes poids s'appliquent à chaque séquence.
- **Softmax et −∞** : exp(−∞) = 0, donc ces positions reçoivent un **poids exactement nul**. Les positions restantes (scores 0) se partagent le poids à égalité, d'où 1/(t+1) : on retrouve la moyenne causale. Le masque empêche ainsi de regarder le futur, et chaque ligne somme toujours à 1.

## Étape 5 : calculer des poids d'attention

In [25]:
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)
head_size = 16

key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)    # (B,T,head_size)
q = query(x)  # (B,T,head_size)
wei = q @ k.transpose(-2, -1)  # (B,T,T)
wei = wei *  head_size**-0.5 # mise a l'echelle
wei = wei.masked_fill( tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v  # (B,T,head_size)

> **Questions d'interprétation** — Quelle ligne de `wei[0]` correspond au dernier token ? Pourquoi la sortie mélange-t-elle v, et non directement k ?

**Réponses**
- **Dernier token** : la **dernière ligne**, `wei[0][-1]` (indice T−1 = 7). C'est la seule sans zéro : le dernier token voit les 8 positions.
- **Pourquoi mélanger v et non k** : les trois projections ont des rôles distincts. `q·k` sert uniquement à calculer **combien** chaque token doit compter (les poids). `v` est **ce qui est transmis** une fois le token choisi. Séparer k et v permet au modèle d'utiliser une propriété pour se faire « trouver » (k) et de transmettre une autre information (v).

## Étape 6 : stabiliser Softmax
La variance du produit scalaire augmente avec d_h. Diviser par √d_h maintient les scores dans une zone où Softmax reste informative.

In [26]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

raw = q @ k.transpose(-2, -1)
scaled = raw * head_size**-0.5

print(raw.var(), scaled.var())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

tensor(19.8478) tensor(1.2405)
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


> **Bilan du TP 2** — Vérifiez la somme des lignes de `wei`, l'absence de poids vers le futur et la forme finale. Expliquez le rôle distinct de Q, K et V.

**Réponses**
- **Vérifications** : `wei[0].sum(dim=-1)` donne des 1 partout. Tout ce qui est au-dessus de la diagonale vaut 0 (aucun poids vers le futur). `out.shape` = `(4, 8, 16)` = `(B, T, d_h)`.
- **Rôle de Q, K, V** :
  - **Q (query)** : ce que le token courant **cherche** ;
  - **K (key)** : ce que chaque token **propose** pour être trouvé. Le produit `q·k` mesure leur compatibilité ;
  - **V (value)** : **l'information transmise**. La sortie est la moyenne des v pondérée par softmax(q·kᵀ/√d_h).
- **Étape 6** : la sortie montre `Var(raw) ≈ 19,8`, du même ordre que d_h = 16, et `Var(scaled) ≈ 1,24 ≈ 1`. (Les valeurs changent un peu à chaque exécution, car `q` et `k` sont tirés au hasard sans graine fixe ; on obtient toujours Var(raw) ≈ d_h et Var(scaled) ≈ 1.) Multiplier z par 8 fait passer le max de 0,29 à 0,80 : sans mise à l'échelle, softmax devient presque un « one-hot », un seul token domine et les gradients vers les autres deviennent quasi nuls, ce qui freine l'apprentissage.